# JiT Diffusion Model Analysis

Load a trained JiT (or Fusion) checkpoint, plot training/validation loss curves,
generate samples, and visualize the x-prediction process as an animated gif.

Set `CHECKPOINT_PATH` below to point at your checkpoint.

In [ ]:
#CHECKPOINT_PATH = "../checkpoints/<run_id>/jit_last.pth"  # <-- edit this

CHECKPOINT_PATH = "../checkpoints/<run_id>/jit_last.pth"

In [ ]:
from pathlib import Path

import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import HTML
from PIL import Image

from jepajitfusion.config import DataConfig
from jepajitfusion.config.decoder_config import DecoderConfig
from jepajitfusion.data.transforms import reverse_transform
from jepajitfusion.decoder.jit_model import JiTModel
from jepajitfusion.decoder.sampler import HeunSampler
from jepajitfusion.utils import get_device

## Load checkpoint and build model

In [ ]:
device = get_device()
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)

print(f"Checkpoint keys: {list(checkpoint.keys())}")
print(f"Epoch: {checkpoint['epoch']}")
print(f"Run ID: {checkpoint.get('run_id', 'N/A')}")

In [ ]:
dec_cfg = DecoderConfig(**checkpoint["decoder_config"]) if "decoder_config" in checkpoint else DecoderConfig()
ds_cfg = DataConfig(**checkpoint["dataset_config"]) if "dataset_config" in checkpoint else DataConfig()

print(f"Decoder: dim={dec_cfg.dim}, depth={dec_cfg.depth}, heads={dec_cfg.num_heads}")
print(f"Dataset: {ds_cfg.name}, img_size={ds_cfg.img_size}")
print(f"Conditioning: {dec_cfg.conditioning_mode}")

# Check if this is a fusion checkpoint (has encoder_config)
is_fusion = "encoder_config" in checkpoint
if is_fusion:
    from jepajitfusion.config.encoder_config import EncoderConfig
    enc_cfg = EncoderConfig(**checkpoint["encoder_config"])
    print(f"Fusion checkpoint — encoder: embed_dim={enc_cfg.embed_dim}, depth={enc_cfg.depth}")

model = JiTModel(
    img_size=ds_cfg.img_size,
    patch_size=dec_cfg.patch_size,
    in_channels=ds_cfg.num_channels,
    dim=dec_cfg.dim,
    depth=dec_cfg.depth,
    num_heads=dec_cfg.num_heads,
    mlp_ratio=dec_cfg.mlp_ratio,
    bottleneck_dim=dec_cfg.bottleneck_dim,
    num_classes=dec_cfg.num_classes,
    conditioning_mode=dec_cfg.conditioning_mode,
    jepa_dim=dec_cfg.jepa_dim if is_fusion else 256,
)

# Prefer EMA weights if available
if "ema_state_dicts" in checkpoint and checkpoint["ema_state_dicts"]:
    model.load_state_dict(checkpoint["ema_state_dicts"][0])
    print("Loaded EMA model weights")
else:
    model.load_state_dict(checkpoint["model_state_dict"])
    print("Loaded model weights")

model = model.to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"JiT model: {n_params:.1f}M parameters on {device}")

## Training and validation loss curves

In [ ]:
train_losses = checkpoint.get("train_losses")
val_losses = checkpoint.get("val_losses")

if train_losses:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    epochs = range(1, len(train_losses) + 1)
    ax1.plot(epochs, train_losses, linewidth=0.8, label="Train")
    if val_losses:
        val_interval = max(1, len(train_losses) // len(val_losses))
        val_epochs = [val_interval * (i + 1) for i in range(len(val_losses))]
        ax1.plot(val_epochs, val_losses, linewidth=0.8, label="Val")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, train_losses, linewidth=0.8, label="Train")
    if val_losses:
        ax2.plot(val_epochs, val_losses, linewidth=0.8, label="Val")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Loss (log scale)")
    ax2.set_title("Loss (log scale)")
    ax2.set_yscale("log")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    pipeline = "Fusion" if is_fusion else "JiT"
    fig.suptitle(
        f"{pipeline} training — {ds_cfg.name} (run {checkpoint.get('run_id', 'N/A')})",
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()

    print(f"Final train loss: {train_losses[-1]:.6f} (epoch {len(train_losses)})")
    print(f"Min train loss:   {min(train_losses):.6f} (epoch {train_losses.index(min(train_losses)) + 1})")
    if val_losses:
        print(f"Min val loss:     {min(val_losses):.6f}")
else:
    print("No train_losses in checkpoint. Re-train to get loss curves.")

## Generate samples

In [ ]:
N_SAMPLES = 16
NUM_STEPS = 50
CFG_SCALE = 1.5
NOISE_SCALE = 0.375
SEED = 1999

In [ ]:
torch.manual_seed(SEED)

sampler = HeunSampler(num_steps=NUM_STEPS, cfg_scale=CFG_SCALE, noise_scale=NOISE_SCALE)
shape = (N_SAMPLES, ds_cfg.num_channels, ds_cfg.img_size, ds_cfg.img_size)

cond = None
uncond_cond = None
if dec_cfg.conditioning_mode == "label" and dec_cfg.num_classes > 0:
    cond = torch.randint(0, dec_cfg.num_classes, (N_SAMPLES,), device=device)
    uncond_cond = torch.full((N_SAMPLES,), dec_cfg.num_classes, device=device, dtype=torch.long)
    print(f"Sampling with label conditioning (classes: {cond.tolist()})")
else:
    print("Sampling unconditionally")

print(f"Generating {N_SAMPLES} samples with {NUM_STEPS} Heun steps...")
samples = sampler.sample(model, shape, device, conditioning=cond, uncond_conditioning=uncond_cond)
print("Done.")

In [ ]:
rev_t = reverse_transform()

cols = min(8, N_SAMPLES)
rows = (N_SAMPLES + cols - 1) // cols
scale = max(1, 256 // ds_cfg.img_size)  # upscale small images for visibility
fig_w = cols * ds_cfg.img_size * scale / 100
fig_h = rows * ds_cfg.img_size * scale / 100

fig, axes = plt.subplots(rows, cols, figsize=(fig_w, fig_h))
axes = np.array(axes).reshape(rows, cols)

for i in range(N_SAMPLES):
    r, c = divmod(i, cols)
    img = rev_t(samples[i].cpu())
    axes[r, c].imshow(img)
    axes[r, c].axis("off")

# Hide unused axes
for i in range(N_SAMPLES, rows * cols):
    r, c = divmod(i, cols)
    axes[r, c].axis("off")

pipeline = "Fusion" if is_fusion else "JiT"
fig.suptitle(
    f"{pipeline} samples — {ds_cfg.name} (epoch {checkpoint['epoch']})",
    fontsize=12,
)
plt.tight_layout()
plt.show()

## Animated x-prediction process

At each Heun step the model directly predicts the clean image $\hat{x}$ from the
current noisy state $z_t$.  We capture these x-predictions throughout sampling to
visualize how the model's "best guess" evolves from noise to the final result.

In [ ]:
GIF_SAMPLES = 8
GIF_STEPS = 50
SAVE_EVERY_K = 2  # capture an x-prediction frame every K steps
GIF_SEED = 42

In [ ]:
@torch.no_grad()
def sample_with_x_predictions(
    model,
    shape,
    device,
    num_steps=50,
    noise_scale=0.375,
    cfg_scale=1.5,
    conditioning=None,
    uncond_conditioning=None,
    save_every_k=1,
):
    """Heun sampling that also returns intermediate x-predictions.

    Returns:
        final: (B, C, H, W) final samples.
        x_preds: list of (B, C, H, W) x-predictions captured every k steps.
        timesteps: list of float timestep values for each captured frame.
    """
    z = torch.randn(shape, device=device) * noise_scale
    x_preds = []
    timesteps = []

    for i in range(num_steps):
        t_val = i / num_steps
        t_next_val = (i + 1) / num_steps
        dt = t_next_val - t_val

        t = torch.full((shape[0],), t_val, device=device)
        t_next = torch.full((shape[0],), t_next_val, device=device)

        # Get x-prediction and velocity
        x_pred = model(z, t, conditioning)
        t_view = t.view(-1, 1, 1, 1)
        denom = (1 - t_view).clamp(min=0.05)
        v = (x_pred - z) / denom

        # Apply CFG if needed
        if cfg_scale > 1.0 and uncond_conditioning is not None:
            x_pred_uncond = model(z, t, uncond_conditioning)
            v_uncond = (x_pred_uncond - z) / denom
            v = v_uncond + cfg_scale * (v - v_uncond)
            # Recompute guided x_pred for visualization
            x_pred = z + v * denom

        # Capture x-prediction
        if i % save_every_k == 0 or i == num_steps - 1:
            x_preds.append(x_pred.cpu())
            timesteps.append(t_val)

        # Euler step
        z_euler = z + dt * v

        # Heun correction (skip at last step)
        if i < num_steps - 1:
            x_pred_next = model(z_euler, t_next, conditioning)
            v_next = (x_pred_next - z_euler) / (1 - t_next.view(-1, 1, 1, 1)).clamp(min=0.05)
            if cfg_scale > 1.0 and uncond_conditioning is not None:
                x_pred_uncond_next = model(z_euler, t_next, uncond_conditioning)
                denom_next = (1 - t_next.view(-1, 1, 1, 1)).clamp(min=0.05)
                v_uncond_next = (x_pred_uncond_next - z_euler) / denom_next
                v_next = v_uncond_next + cfg_scale * (v_next - v_uncond_next)
            v_avg = 0.5 * (v + v_next)
            z = z + dt * v_avg
        else:
            z = z_euler

    return z, x_preds, timesteps

In [ ]:
torch.manual_seed(GIF_SEED)

gif_shape = (GIF_SAMPLES, ds_cfg.num_channels, ds_cfg.img_size, ds_cfg.img_size)

gif_cond = None
gif_uncond = None
if dec_cfg.conditioning_mode == "label" and dec_cfg.num_classes > 0:
    gif_cond = torch.randint(0, dec_cfg.num_classes, (GIF_SAMPLES,), device=device)
    gif_uncond = torch.full((GIF_SAMPLES,), dec_cfg.num_classes, device=device, dtype=torch.long)

print(f"Generating {GIF_SAMPLES} samples with {GIF_STEPS} steps, capturing every {SAVE_EVERY_K}...")
final, x_preds, timesteps = sample_with_x_predictions(
    model, gif_shape, device,
    num_steps=GIF_STEPS,
    noise_scale=NOISE_SCALE,
    cfg_scale=CFG_SCALE,
    conditioning=gif_cond,
    uncond_conditioning=gif_uncond,
    save_every_k=SAVE_EVERY_K,
)
print(f"Captured {len(x_preds)} frames at timesteps: {[f'{t:.2f}' for t in timesteps]}")

In [ ]:
def combine_images(images, rows, cols):
    """Tile PIL images into a single grid."""
    w, h = images[0].size
    grid = Image.new("RGB", (cols * w, rows * h))
    for i, img in enumerate(images):
        grid.paste(img, ((i % cols) * w, (i // cols) * h))
    return grid


rev_t = reverse_transform()
gif_cols = min(8, GIF_SAMPLES)
gif_rows = (GIF_SAMPLES + gif_cols - 1) // gif_cols
scale = max(1, 256 // ds_cfg.img_size)

# Build a grid image for each captured timestep
frames = []
for x_pred in x_preds:
    pil_images = [rev_t(x_pred[i]) for i in range(GIF_SAMPLES)]
    grid = combine_images(pil_images, gif_rows, gif_cols)
    if scale > 1:
        grid = grid.resize((grid.width * scale, grid.height * scale), Image.NEAREST)
    frames.append(grid)

print(f"Built {len(frames)} grid frames ({frames[0].size[0]}x{frames[0].size[1]} px)")

In [ ]:
fig = plt.figure(
    figsize=(frames[0].size[0] / 100, frames[0].size[1] / 100), dpi=100
)
ax = plt.gca()
ax.axis("off")

ims = []
for i, frame in enumerate(frames):
    title = ax.text(
        0.5, 1.02, f"t = {timesteps[i]:.2f}",
        ha="center", va="bottom", transform=ax.transAxes, fontsize=12,
    )
    im = plt.imshow(frame)
    ims.append([im, title])

ani = animation.ArtistAnimation(
    fig, ims, interval=200, repeat_delay=3000, repeat=True
)
plt.tight_layout()

In [ ]:
# Save gif
output_dir = Path("../samples")
output_dir.mkdir(exist_ok=True, parents=True)

pipeline = "fusion" if is_fusion else "jit"
gif_path = output_dir / f"{pipeline}_x_prediction_{ds_cfg.name}.gif"
ani.save(str(gif_path), writer="pillow", fps=5)
print(f"Saved gif to {gif_path}")

In [ ]:
# Play inline
plt.rcParams["animation.embed_limit"] = 100  # MB
HTML(ani.to_jshtml(default_mode="once"))

## x-prediction filmstrip

Static view: show the model's x-prediction $\hat{x}$ for one sample across evenly-spaced timesteps.

In [ ]:
# Pick ~8 evenly spaced frames for a filmstrip
n_filmstrip = min(8, len(x_preds))
filmstrip_idxs = np.linspace(0, len(x_preds) - 1, n_filmstrip, dtype=int)
sample_idx = 0  # which sample to show

fig, axes = plt.subplots(1, n_filmstrip, figsize=(2 * n_filmstrip, 2.5))

for col, fi in enumerate(filmstrip_idxs):
    img = rev_t(x_preds[fi][sample_idx])
    axes[col].imshow(img)
    axes[col].set_title(f"t={timesteps[fi]:.2f}", fontsize=9)
    axes[col].axis("off")

fig.suptitle("x-prediction across sampling steps", fontsize=12)
plt.tight_layout()
plt.show()